# Create GVA Awards (Generalitat Valenciana)

Creates Generalitat Valenciana research/R&D+i awards from the GVA open-data bulk subsidies files.

**Prerequisites:** run `scripts/local/gva_to_s3.py` to download + filter + aggregate + upload first.

**Data source:** CKAN datasets `eco-gvo-subv-2022`..`eco-gvo-subv-2026` on dadesobertes.gva.es ("Ayudas y subvenciones concedidas por la Generalitat Valenciana", monthly CSVs; catalogued nationally as datos.gob.es `a10002983-...`). GVA only publishes the 4 years following each concession year, so the corpus is a rolling ~5-year window.

**INCLUSION RULE (research scoping, applied in the script):** keep a subsidy iff (a) `cd_finalidad = '17'` ("Investigación, desarrollo e innovación" budget policy), OR (b) the convocatoria/título/línea/concedente text matches the research regex (predoctoral/postdoctoral fellowships, PROMETEO, GenT, Santiago Grisolía, ACIF, research-group and scientific-infrastructure calls, Agència Valenciana de la Innovació programmes), minus an exclusion regex (retail "innovación comercial", school "innovación educativa", "cooperación educativa" student placements, DANA flood aid); ministry names embedding "Investigación" (2015-2019 education conselleria) are stripped before matching.

**S3 location:** `s3a://openalex-ingest/awards/gva/gva_projects.parquet`

**GVA funder in OpenAlex:** funder_id 4320321864 · display_name "Generalitat Valenciana" · ROR https://ror.org/0097mvx21 · doi 10.13039/501100003359 · ES.

**Schema notes:**
- **Amounts are EUR whole units** (`importe`; no minor-unit encoding). Aggregated per award key.
- `funder_award_id` = `{cod_convocatoria}:{beneficiario NIF}` — the source has **no per-grant id**; the call code + beneficiary tax id is the natural key, aggregated in the script (sum importe, min fecha_concesion), so it is collision-free by construction.
- **Anonymized person rows are EXCLUDED at scrape time (AGAUR precedent):** GVA publishes physical-person beneficiaries GDPR-redacted (empty NIF, `nombre` = "PERSONA FÍSICA QUE (NO) DESARROLLA ACTIVIDAD ECONÓMICA"), so individual fellowships awarded to the fellow are unusable as grant records. All shipped beneficiaries are **named institutions/companies** and map to `lead_investigator.affiliation.name` only — no PI names exist in the source (`pct_with_pi` expected 0%).
- `start_date` = earliest concession date; grants are single-dated in the register (no end dates published).

provenance `gva`, priority 419.


## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.gva_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/gva/gva_projects.parquet`;


In [ ]:
%sql
SELECT COUNT(*) as total_awards FROM openalex.awards.gva_raw;

In [ ]:
%sql
DESCRIBE openalex.awards.gva_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.gva_raw LIMIT 5;

## Step 1.6: Funder existence fail-fast

Must return exactly 1 row (F4320321864 is Crossref-registered / Path A). If 0, STOP.

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi
FROM openalex.common.funder
WHERE funder_id = 4320321864;


## Step 2: Create GVA Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.gva_awards
USING delta
AS
WITH
gva_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320321864  -- Generalitat Valenciana
),
awards_transformed AS (
    SELECT
        abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.funder_award_id)))) % 9000000000 as id,
        COALESCE(NULLIF(TRIM(g.titulo_extracto), ''), g.convocatoria) as display_name,
        CAST(NULL AS STRING) as description,
        f.funder_id,
        g.funder_award_id,
        CASE WHEN TRY_CAST(g.amount AS DOUBLE) > 0 THEN TRY_CAST(g.amount AS DOUBLE) ELSE NULL END as amount,
        CASE WHEN TRY_CAST(g.amount AS DOUBLE) > 0 THEN 'EUR' ELSE NULL END as currency,
        struct(
            CONCAT('https://openalex.org/F', f.funder_id) as id,
            f.display_name, f.ror_id, f.doi
        ) as funder,
        CASE
            WHEN LOWER(g.convocatoria) RLIKE '(predoctoral|postdoctoral|posdoctoral|doctorand|beca|contractaci|contrataci|grisol|acif|apostd)' THEN 'fellowship'
            ELSE 'research'
        END as funding_type,
        COALESCE(NULLIF(TRIM(g.linea), ''), NULLIF(TRIM(g.linea_agregada), ''), g.finalidad) as funder_scheme,
        'gva' as provenance,
        TRY_TO_DATE(g.start_date, 'yyyy-MM-dd') as start_date,
        CAST(NULL AS DATE) as end_date,
        COALESCE(YEAR(TRY_TO_DATE(g.start_date, 'yyyy-MM-dd')), TRY_CAST(g.ejercicio AS INT)) as start_year,
        CAST(NULL AS INT) as end_year,
        -- GVA publishes no PI names (anonymized persons excluded at scrape
        -- time); beneficiaries are named institutions -> affiliation only.
        CASE
            WHEN g.institution_name IS NOT NULL AND TRIM(g.institution_name) != '' THEN
                struct(
                    CAST(NULL AS STRING) as given_name,
                    CAST(NULL AS STRING) as family_name,
                    CAST(NULL AS STRING) as orcid,
                    CAST(NULL AS DATE) as role_start,
                    struct(
                        g.institution_name as name,
                        'Spain' as country,
                        CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) as ids
                    ) as affiliation
                )
            ELSE NULL
        END as lead_investigator,
        CAST(NULL AS STRUCT<
            given_name:STRING, family_name:STRING, orcid:STRING,
            role_start:DATE, affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >) as co_lead_investigator,
        CAST(NULL AS ARRAY<STRUCT<
            given_name:STRING, family_name:STRING, orcid:STRING,
            role_start:DATE, affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >>) as investigators,
        COALESCE(NULLIF(TRIM(g.url_publi), ''), NULLIF(TRIM(g.url_base), ''), 'https://gvaoberta.gva.es/es/buscador-de-subvencions') as landing_page_url,
        CAST(NULL AS STRING) as doi
    FROM openalex.awards.gva_raw g
    CROSS JOIN gva_funder f
)
SELECT *,
    concat('https://api.openalex.org/works?filter=awards.id:G', id) as works_api_url,
    current_timestamp() as created_date,
    current_timestamp() as updated_date
FROM awards_transformed;


## Step 3: Insert into openalex_awards_raw (priority 419)

In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'gva' AND priority = 419;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    419 as priority
FROM openalex.awards.gva_awards;


## Verification Queries

In [ ]:
%sql
SELECT COUNT(*) as total_gva_awards FROM openalex.awards.gva_awards;

In [ ]:
%sql
SELECT funder_award_id, display_name, funding_type, amount, currency, start_year,
       lead_investigator.family_name, lead_investigator.affiliation.name
FROM openalex.awards.gva_awards LIMIT 10;


In [ ]:
%sql
SELECT funding_type, COUNT(*) as cnt FROM openalex.awards.gva_awards
GROUP BY funding_type ORDER BY cnt DESC;


In [ ]:
%sql
-- §6.3 completeness. pct_with_amount expected >95% (importe published for
-- essentially every concession). NOTE: pct_with_pi expected 0% — GVA
-- anonymizes person beneficiaries and publishes no PI names; institutions
-- carry affiliation only (AGAUR pattern).
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(amount) as has_amount,
    COUNT(lead_investigator.family_name) as has_pi,
    COUNT(lead_investigator.affiliation.name) as has_institution,
    COUNT(start_date) as has_start_date,
    ROUND(try_divide(COUNT(amount) * 100.0, COUNT(*)), 1) as pct_with_amount,
    ROUND(try_divide(COUNT(lead_investigator.affiliation.name) * 100.0, COUNT(*)), 1) as pct_with_institution
FROM openalex.awards.gva_awards;


In [ ]:
%sql
-- §6.7 amount/currency coverage (FAIL-FAST)
SELECT COUNT(*) AS total, COUNT(amount) AS has_amount,
    ROUND(COUNT(amount) * 100.0 / COUNT(*), 1) AS pct_amount,
    COUNT(DISTINCT currency) AS distinct_currencies, collect_set(currency) AS currencies,
    MIN(amount) AS min_amount, MAX(amount) AS max_amount, AVG(amount) AS avg_amount
FROM openalex.awards.gva_awards;


In [ ]:
%sql
SELECT start_year, COUNT(*) as cnt FROM openalex.awards.gva_awards
WHERE start_year IS NOT NULL GROUP BY start_year ORDER BY start_year DESC LIMIT 20;


In [ ]:
%sql
SELECT lead_investigator.affiliation.name as institution, COUNT(*) as grant_count
FROM openalex.awards.gva_awards WHERE lead_investigator.affiliation.name IS NOT NULL
GROUP BY 1 ORDER BY 2 DESC LIMIT 20;
